In [0]:
#### Loading libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

### Question1: 

### Level:

**Basic**

### Question:

You are given an employee dataset containing employee details.

**Task:**
Using PySpark, select only the following columns:

* `employee_id`
* `employee_name`
* `department`
* `salary`

Then display the resulting DataFrame.

### Practical Use Case:

In a real-world ETL pipeline, source systems often contain many columns, while downstream processes require only a subset. Selecting only the required columns is a common **data transformation and column-pruning** step.

### PySpark Query for Table Creation:

```python
data = [
    (101, "Rahul", "IT", 65000, "Delhi"),
    (102, "Priya", "HR", 55000, "Mumbai"),
    (103, "Amit", "Finance", 72000, "Pune"),
    (104, "Sneha", "IT", 68000, "Bangalore"),
    (105, "Arjun", "Sales", 50000, "Hyderabad")
]

columns = [
    "employee_id",
    "employee_name",
    "department",
    "salary",
    "city"
]

df = spark.createDataFrame(data, columns)

df.display()
```

### Expected Output:

| employee_id | employee_name | department | salary |
| ----------: | ------------- | ---------- | -----: |
|         101 | Rahul         | IT         |  65000 |
|         102 | Priya         | HR         |  55000 |
|         103 | Amit          | Finance    |  72000 |
|         104 | Sneha         | IT         |  68000 |
|         105 | Arjun         | Sales      |  50000 |

**Your task:** Write the PySpark transformation that produces this output.

In [0]:
## Question 1:
data = [
    (101, "Rahul", "IT", 65000, "Delhi"),
    (102, "Priya", "HR", 55000, "Mumbai"),
    (103, "Amit", "Finance", 72000, "Pune"),
    (104, "Sneha", "IT", 68000, "Bangalore"),
    (105, "Arjun", "Sales", 50000, "Hyderabad")
]

columns = [
    "employee_id",
    "employee_name",
    "department",
    "salary",
    "city"
]

df = spark.createDataFrame(data, columns)

df.display()

In [0]:
## Answer 1:
df.select(col("employee_id"), col("employee_name"), col("department"), col("salary")).display()

**Question 1**

**Level:**
Basic

**Question:**
You have just ingested a raw file containing employee records into a PySpark DataFrame. Before writing this data to our curated tables, you need to standardize the schema.
Write a PySpark query to select only the `emp_id`, `full_name`, and `department` columns. While selecting them, rename the `full_name` column to `employee_name` to match our target database schema.

**Practical Use Case:**
In a real-world ETL pipeline, raw data (Bronze layer) often arrives with inconsistent or messy column names from various source systems (like APIs or CSVs). Renaming columns and selecting only the required fields is one of the most fundamental first steps when moving data to a standardized Silver layer.

**PySpark Query for Table Creation:**

```python
data = [
    (1001, "Alice Smith", "Data Engineering", 110000, "2021-06-15"),
    (1002, "Bob Jones", "Marketing", 75000, "2022-03-10"),
    (1003, "Charlie Brown", "Sales", 85000, "2020-11-25"),
    (1004, "Diana Prince", "Data Engineering", 125000, "2019-08-01")
]

columns = ["emp_id", "full_name", "department", "salary", "hire_date"]

df = spark.createDataFrame(data, columns)
df.display()

```

**Expected Output:**

| emp_id | employee_name | department |
| --- | --- | --- |
| 1001 | Alice Smith | Data Engineering |
| 1002 | Bob Jones | Marketing |
| 1003 | Charlie Brown | Sales |
| 1004 | Diana Prince | Data Engineering |


In [0]:
## Question 1: 
data = [
    (1001, "Alice Smith", "Data Engineering", 110000, "2021-06-15"),
    (1002, "Bob Jones", "Marketing", 75000, "2022-03-10"),
    (1003, "Charlie Brown", "Sales", 85000, "2020-11-25"),
    (1004, "Diana Prince", "Data Engineering", 125000, "2019-08-01")
]

columns = ["emp_id", "full_name", "department", "salary", "hire_date"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
## Answer 1:
df.select(col("emp_id"), col("full_name").alias("employee_name"), col("department")).display()

### Question 2

**Level:**
Basic

**Question:**
You are building a Data Quality (DQ) check for the next step of the pipeline. Write a PySpark query to **filter out (remove)** any records where the `department` is NULL OR the `salary` is less than 50,000.

**Practical Use Case:**
In a real-world Data Engineering pipeline, filtering out "bad" or incomplete data is a critical step when moving data from the Bronze (raw) layer to the Silver (cleansed) layer. You must ensure that downstream analytics dashboards do not break due to missing dimensions (like a NULL department) or invalid metrics (like negative/abnormally low salaries).

**PySpark Query for Table Creation:**

```python
data = [
    (1, "Alice", "Engineering", 120000),
    (2, "Bob", None, 95000),          # Invalid: NULL department
    (3, "Charlie", "Sales", 45000),   # Invalid: Salary < 50000
    (4, "Diana", "Marketing", 85000),
    (5, "Eve", "Engineering", -5000)  # Invalid: Salary < 50000
]

columns = ["emp_id", "employee_name", "department", "salary"]

df = spark.createDataFrame(data, columns)
df.display()

```

**Expected Output:**

| emp_id | employee_name | department | salary |
| --- | --- | --- | --- |
| 1 | Alice | Engineering | 120000 |
| 4 | Diana | Marketing | 85000 |

In [0]:
## Question:
data = [
    (1, "Alice", "Engineering", 120000),
    (2, "Bob", None, 95000),          # Invalid: NULL department
    (3, "Charlie", "Sales", 45000),   # Invalid: Salary < 50000
    (4, "Diana", "Marketing", 85000),
    (5, "Eve", "Engineering", -5000)  # Invalid: Salary < 50000
]

columns = ["emp_id", "employee_name", "department", "salary"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
## Answer:
df.filter((col("department").isNotNull()) & (col("salary") >= 50000)).display()

### Question 3

**Level:**
Basic

**Question:**
You are processing a batch of daily transactions. Due to an upstream system glitch, some transactions were sent multiple times, resulting in duplicate rows.
Write a PySpark query to remove these duplicates, ensuring that each `transaction_id` appears only once in the final DataFrame.

**Practical Use Case:**
Idempotency and deduplication are foundational to ETL. Upstream source systems (like Kafka, REST APIs, or transactional databases) frequently resend data due to network timeouts or retries. A robust Data Engineering pipeline must gracefully handle duplicate data to avoid double-counting revenue or inflating metrics.

**PySpark Query for Table Creation:**

```python
data = [
    ("TXN-101", "user_1", 150.00, "2023-10-01"),
    ("TXN-102", "user_2", 200.50, "2023-10-01"),
    ("TXN-101", "user_1", 150.00, "2023-10-01"), # Exact duplicate
    ("TXN-103", "user_3", 75.25,  "2023-10-01"),
    ("TXN-102", "user_2", 200.50, "2023-10-01"), # Exact duplicate
    ("TXN-104", "user_1", 50.00,  "2023-10-01")
]

columns = ["transaction_id", "user_id", "amount", "transaction_date"]

df = spark.createDataFrame(data, columns)
df.display()

```

**Expected Output:**
*(Note: The order of the output rows does not matter, as Spark processes data in a distributed manner.)*

| transaction_id | user_id | amount | transaction_date |
| --- | --- | --- | --- |
| TXN-101 | user_1 | 150.00 | 2023-10-01 |
| TXN-102 | user_2 | 200.50 | 2023-10-01 |
| TXN-103 | user_3 | 75.25 | 2023-10-01 |
| TXN-104 | user_1 | 50.00 | 2023-10-01 |

In [0]:
## Question:
data = [
    ("TXN-101", "user_1", 150.00, "2023-10-01"),
    ("TXN-102", "user_2", 200.50, "2023-10-01"),
    ("TXN-101", "user_1", 150.00, "2023-10-01"), # Exact duplicate
    ("TXN-103", "user_3", 75.25,  "2023-10-01"),
    ("TXN-102", "user_2", 200.50, "2023-10-01"), # Exact duplicate
    ("TXN-104", "user_1", 50.00,  "2023-10-01")
]

columns = ["transaction_id", "user_id", "amount", "transaction_date"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
## Answer:
df.dropDuplicates(subset=["transaction_id"]).display()

### Question 4

**Level:**
Basic

**Question:**
You are processing a user dataset. The business wants to classify users into loyalty tiers based on their reward points.
Write a PySpark query to create a **new column** called `loyalty_tier`.

* If a user has **5000 or more** `reward_points`, their tier should be **"Gold"**.
* Otherwise, their tier should be **"Standard"**.

**Practical Use Case:**
Deriving new business columns (calculated fields) is a core part of the "Transform" phase in ETL. Data Engineers frequently map raw, continuous numerical data into discrete categorical buckets so that Data Analysts can easily build dashboards (e.g., grouping sales by "High Value" vs "Low Value" customers).

**PySpark Query for Table Creation:**

```python
from pyspark.sql.functions import col, when, lit

data = [
    (1, "Alice", 7500),
    (2, "Bob", 2000),
    (3, "Charlie", 5000),
    (4, "Diana", 4999),
    (5, "Eve", 12000)
]

columns = ["user_id", "user_name", "reward_points"]

df = spark.createDataFrame(data, columns)
df.display()

```

**Expected Output:**

| user_id | user_name | reward_points | loyalty_tier |
| --- | --- | --- | --- |
| 1 | Alice | 7500 | Gold |
| 2 | Bob | 2000 | Standard |
| 3 | Charlie | 5000 | Gold |
| 4 | Diana | 4999 | Standard |
| 5 | Eve | 12000 | Gold |

*Waiting for your PySpark solution.*

In [0]:
## Question:
from pyspark.sql.functions import col, when, lit

data = [
    (1, "Alice", 7500),
    (2, "Bob", 2000),
    (3, "Charlie", 5000),
    (4, "Diana", 4999),
    (5, "Eve", 12000)
]

columns = ["user_id", "user_name", "reward_points"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
## Answer:
df.withColumn('loyalty_tier', when(col('reward_points')>=5000, 'Gold').otherwise('Standard')).display()

### Question 5

**Level:**
Basic

**Question:**
You are processing a batch of e-commerce orders. The reporting team wants a summary table showing the performance of each product category.
Write a PySpark query to group the data by `category`, and calculate two new columns:

1. `total_sales`: The sum of the `sales_amount` for that category.
2. `order_count`: The total number of orders placed in that category.

**Practical Use Case:**
Aggregation is a critical component of the "Transform" phase. Data Engineers frequently roll up high-volume, granular transaction data (Bronze/Silver layer) into aggregated summary tables (Gold layer). These Gold tables are then connected directly to BI tools (like Tableau or PowerBI) because querying pre-aggregated data is drastically faster and cheaper than computing it on the fly.

**PySpark Query for Table Creation:**

```python
from pyspark.sql.functions import col, sum, count

data = [
    ("ORD-001", "Electronics", 500.00),
    ("ORD-002", "Clothing", 150.00),
    ("ORD-003", "Electronics", 250.00),
    ("ORD-004", "Home & Garden", 120.00),
    ("ORD-005", "Clothing", 90.00),
    ("ORD-006", "Electronics", 300.00)
]

columns = ["order_id", "category", "sales_amount"]

df = spark.createDataFrame(data, columns)
df.display()

```

**Expected Output:**
*(Note: The order of rows may vary due to distributed processing)*

| category | total_sales | order_count |
| --- | --- | --- |
| Electronics | 1050.00 | 3 |
| Clothing | 240.00 | 2 |
| Home & Garden | 120.00 | 1 |

*Waiting for your PySpark solution.*

In [0]:
## Question:
from pyspark.sql.functions import col, sum, count

data = [
    ("ORD-001", "Electronics", 500.00),
    ("ORD-002", "Clothing", 150.00),
    ("ORD-003", "Electronics", 250.00),
    ("ORD-004", "Home & Garden", 120.00),
    ("ORD-005", "Clothing", 90.00),
    ("ORD-006", "Electronics", 300.00)
]

columns = ["order_id", "category", "sales_amount"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
## Answer:
df.groupBy('category').agg(sum(col('sales_amount')).alias('total_sales'), count(col('order_id')).alias('order_count')).display()


### Question 6

**Level:**
Basic

**Question:**
You are processing a user profile dataset. Unfortunately, the upstream system does not enforce strict validation, so some records have missing (NULL) values for `country` and `age`.
Write a PySpark query to clean this DataFrame:

1. Replace any NULL values in the `country` column with the string **"Unknown"**.
2. Replace any NULL values in the `age` column with the integer **0**.

**Practical Use Case:**
Handling missing data is a daily task in ETL. Downstream machine learning models or analytical functions often fail if they encounter NULL values. A standard step in Silver-layer processing is to apply default values (imputation) to missing fields so the data remains usable.

**PySpark Query for Table Creation:**

```python
data = [
    (1, "Alice", "USA", 28),
    (2, "Bob", None, 35),           # Missing country
    (3, "Charlie", "UK", None),     # Missing age
    (4, "Diana", None, None),       # Missing both
    (5, "Eve", "Canada", 42)
]

columns = ["customer_id", "name", "country", "age"]

df = spark.createDataFrame(data, columns)
df.display()

```

**Expected Output:**

| customer_id | name | country | age |
| --- | --- | --- | --- |
| 1 | Alice | USA | 28 |
| 2 | Bob | Unknown | 35 |
| 3 | Charlie | UK | 0 |
| 4 | Diana | Unknown | 0 |
| 5 | Eve | Canada | 42 |

*Waiting for your PySpark solution.*

In [0]:
## Question:
data = [
    (1, "Alice", "USA", 28),
    (2, "Bob", None, 35),           # Missing country
    (3, "Charlie", "UK", None),     # Missing age
    (4, "Diana", None, None),       # Missing both
    (5, "Eve", "Canada", 42)
]

columns = ["customer_id", "name", "country", "age"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
## Answer
df.fillna({"country": "Unknown", "age": 0}).display()

### Question 7

**Level:**
Basic

**Question:**
You are cleaning a user profile dataset. The names are messy—some are lowercase, some are uppercase—and they are split into two columns.
Write a PySpark query that does the following:

1. Creates a new column called `full_name` by combining `first_name` and `last_name` with a space in between.
2. Converts the entire `full_name` string to **UPPERCASE**.
3. **Drops** the original `first_name` and `last_name` columns from the final DataFrame.

**Practical Use Case:**
String manipulation and standardizing text formats (like names, addresses, or product codes) is a classic Bronze-to-Silver data cleansing step. Furthermore, dropping unnecessary columns after deriving new ones reduces storage costs and memory overhead in downstream processing.

**PySpark Query for Table Creation:**

```python
from pyspark.sql.functions import col, concat_ws, upper

data = [
    (101, "john", "doe"),
    (102, "JANE", "SMITH"),
    (103, "michael", "Jordan"),
    (104, "Alice", "wonderland")
]

columns = ["user_id", "first_name", "last_name"]

df = spark.createDataFrame(data, columns)
df.display()

```

**Expected Output:**

| user_id | full_name |
| --- | --- |
| 101 | JOHN DOE |
| 102 | JANE SMITH |
| 103 | MICHAEL JORDAN |
| 104 | ALICE WONDERLAND |

*Waiting for your PySpark solution.*

In [0]:
## Question:
from pyspark.sql.functions import col, concat_ws, upper

data = [
    (101, "john", "doe"),
    (102, "JANE", "SMITH"),
    (103, "michael", "Jordan"),
    (104, "Alice", "wonderland")
]

columns = ["user_id", "first_name", "last_name"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
## Answer:
df.withColumn('full_name', upper(concat_ws(' ', col('first_name'), col('last_name')))).select('user_id', 'full_name').display()

### Question 8

**Level:**
Basic

**Question:**
You are processing a sales feed where the date was provided as a string in the format `"yyyy-MM-dd"`.
Write a PySpark query to do the following:

1. Create a new column named `actual_date` by converting the `txn_date_str` string column into a proper PySpark **DateType**.
2. Create another new column named `txn_year` that extracts just the **year** (as an integer) from that date.

**Practical Use Case:**
Time-series data almost always requires casting strings to proper Date or Timestamp types. Furthermore, extracting the Year, Month, or Day is a mandatory step for **Partitioning**. When writing data to a Data Lake (like S3 or ADLS) using Delta or Parquet, Data Engineers heavily rely on these extracted year/month columns to partition the folders for faster querying downstream.

**PySpark Query for Table Creation:**

```python
data = [
    ("ORD-100", "2023-01-15", 150.00),
    ("ORD-101", "2023-11-20", 200.50),
    ("ORD-102", "2024-05-10", 99.99),
    ("ORD-103", "2024-12-25", 500.00)
]

columns = ["order_id", "txn_date_str", "amount"]

df = spark.createDataFrame(data, columns)
df.display()

```

**Expected Output:**

| order_id | txn_date_str | amount | actual_date | txn_year |
| --- | --- | --- | --- | --- |
| ORD-100 | 2023-01-15 | 150.0 | 2023-01-15 | 2023 |
| ORD-101 | 2023-11-20 | 200.5 | 2023-11-20 | 2023 |
| ORD-102 | 2024-05-10 | 99.99 | 2024-05-10 | 2024 |
| ORD-103 | 2024-12-25 | 500.0 | 2024-12-25 | 2024 |

*(Note: `actual_date` must be a DateType, and `txn_year` must be an integer, not strings).*

*Waiting for your PySpark solution.*

In [0]:
## Question:
data = [
    ("ORD-100", "2023-01-15", 150.00),
    ("ORD-101", "2023-11-20", 200.50),
    ("ORD-102", "2024-05-10", 99.99),
    ("ORD-103", "2024-12-25", 500.00)
]

columns = ["order_id", "txn_date_str", "amount"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
## Answer:
df.withColumn('actual_date', to_date(col('txn_date_str')))\
    .withColumn('txn_year', year(col('actual_date')))\
        .display()

### Question 9

**Level:**
Basic

**Question:**
You have two DataFrames: `df_orders` (containing transaction data) and `df_customers` (containing customer profiles).
Write a PySpark query to join these two DataFrames so that you can see the `customer_name` next to their orders.

* Only include orders where a matching customer exists (an Inner Join).
* Your final DataFrame should **only** contain the following columns: `order_id`, `customer_name`, and `amount`.

**Practical Use Case:**
Joining datasets is the heart of dimensional modeling and ETL. You are taking "Fact" data (the orders) and enriching it with "Dimension" data (the customer details) to create a denormalized dataset that is easy for business analysts to query.

**PySpark Query for Table Creation:**

```python
# Create Orders DataFrame (Fact)
data_orders = [
    (1, 101, 250.00), 
    (2, 102, 300.00), 
    (3, 109, 50.00)   # Customer 109 does NOT exist in the customers table
]
cols_orders = ["order_id", "customer_id", "amount"]
df_orders = spark.createDataFrame(data_orders, cols_orders)

# Create Customers DataFrame (Dimension)
data_customers = [
    (101, "Alice"), 
    (102, "Bob"), 
    (103, "Charlie")  # Charlie has no orders
]
cols_customers = ["customer_id", "customer_name"]
df_customers = spark.createDataFrame(data_customers, cols_customers)

print("Orders:")
df_orders.show()
print("Customers:")
df_customers.show()

```

**Expected Output:**

| order_id | customer_name | amount |
| --- | --- | --- |
| 1 | Alice | 250.0 |
| 2 | Bob | 300.0 |

*(Notice that order 3 is dropped because customer 109 is missing, and Charlie is dropped because he has no orders).*

*Waiting for your PySpark solution.*

In [0]:
## Question: 
# Create Orders DataFrame (Fact)
data_orders = [
    (1, 101, 250.00), 
    (2, 102, 300.00), 
    (3, 109, 50.00)   # Customer 109 does NOT exist in the customers table
]
cols_orders = ["order_id", "customer_id", "amount"]
df_orders = spark.createDataFrame(data_orders, cols_orders)

# Create Customers DataFrame (Dimension)
data_customers = [
    (101, "Alice"), 
    (102, "Bob"), 
    (103, "Charlie")  # Charlie has no orders
]
cols_customers = ["customer_id", "customer_name"]
df_customers = spark.createDataFrame(data_customers, cols_customers)

print("Orders:")
df_orders.show()
print("Customers:")
df_customers.show()

In [0]:
## Answer:
df_orders.join(df_customers, df_orders.customer_id == df_customers.customer_id, 'inner')\
    .select('order_id', 'customer_name', 'amount')\
        .display()

### Question 10

**Level:**
Basic

**Question:**
The Marketing team wants to send a promotional email to users who signed up but have **never** placed an order.
Using the same tables as before, write a PySpark query to find these inactive customers.

* Your final DataFrame should **only** contain the `customer_name`.

**Practical Use Case:**
This is a classic "Anti-Join" scenario. Data Engineers frequently need to find records that exist in Table A but are entirely missing from Table B. This is used for finding churned users, identifying orphaned records (data integrity checks), or generating targeted marketing lists.

**PySpark Query for Table Creation:**

```python
from pyspark.sql.functions import col

# Create Orders DataFrame (Fact)
data_orders = [
    (1, 101, 250.00), 
    (2, 102, 300.00)
]
cols_orders = ["order_id", "customer_id", "amount"]
df_orders = spark.createDataFrame(data_orders, cols_orders)

# Create Customers DataFrame (Dimension)
data_customers = [
    (101, "Alice"), 
    (102, "Bob"), 
    (103, "Charlie"), # Charlie has NO orders
    (104, "Diana")    # Diana has NO orders
]
cols_customers = ["customer_id", "customer_name"]
df_customers = spark.createDataFrame(data_customers, cols_customers)

print("Orders:")
df_orders.show()
print("Customers:")
df_customers.show()

```

**Expected Output:**

| customer_name |
| --- |
| Charlie |
| Diana |

*Waiting for your PySpark solution.*

In [0]:
## Questions:
from pyspark.sql.functions import col

# Create Orders DataFrame (Fact)
data_orders = [
    (1, 101, 250.00), 
    (2, 102, 300.00)
]
cols_orders = ["order_id", "customer_id", "amount"]
df_orders = spark.createDataFrame(data_orders, cols_orders)

# Create Customers DataFrame (Dimension)
data_customers = [
    (101, "Alice"), 
    (102, "Bob"), 
    (103, "Charlie"), # Charlie has NO orders
    (104, "Diana")    # Diana has NO orders
]
cols_customers = ["customer_id", "customer_name"]
df_customers = spark.createDataFrame(data_customers, cols_customers)

print("Orders:")
df_orders.show()
print("Customers:")
df_customers.show()

In [0]:
## Answer:
df_customers.join(df_orders, 'customer_id', 'left_anti').select(col('customer_name')).display()

### Question 11

**Level:**
Basic

**Question:**
You are ingesting daily sales data from two different regional teams (US and EU) into a centralized Data Lake. Both teams provide the same data, but the EU team exports their CSV files with the columns in a **different order** than the US team.

Write a PySpark query to combine these two DataFrames (`df_us` and `df_eu`) into a single DataFrame. The combined DataFrame must maintain data integrity (i.e., amounts should stay in the amount column, regions in the region column, etc.).

**Practical Use Case:**
Appending data vertically is a daily ETL task, especially when combining historical data or merging feeds from different third-party vendors. However, relying on the physical position of columns is a massive trap in Big Data processing. You must ensure you are combining data based on the column's *meaning* (its name), not just its index position.

**PySpark Query for Table Creation:**

```python
# US Region Data
data_us = [
    ("ORD-1", 100.00, "US"),
    ("ORD-2", 250.50, "US")
]
cols_us = ["order_id", "amount", "region"]
df_us = spark.createDataFrame(data_us, cols_us)

# EU Region Data (Notice the column order is different)
data_eu = [
    ("EU", "ORD-3", 300.00),
    ("EU", "ORD-4", 150.00)
]
cols_eu = ["region", "order_id", "amount"]
df_eu = spark.createDataFrame(data_eu, cols_eu)

print("US Data:")
df_us.show()
print("EU Data:")
df_eu.show()

```

**Expected Output:**
*(Row order may vary)*

| order_id | amount | region |
| --- | --- | --- |
| ORD-1 | 100.0 | US |
| ORD-2 | 250.5 | US |
| ORD-3 | 300.0 | EU |
| ORD-4 | 150.0 | EU |

*Waiting for your PySpark solution.*

In [0]:
## Question:
# US Region Data
data_us = [
    ("ORD-1", 100.00, "US"),
    ("ORD-2", 250.50, "US")
]
cols_us = ["order_id", "amount", "region"]
df_us = spark.createDataFrame(data_us, cols_us)

# EU Region Data (Notice the column order is different)
data_eu = [
    ("EU", "ORD-3", 300.00),
    ("EU", "ORD-4", 150.00)
]
cols_eu = ["region", "order_id", "amount"]
df_eu = spark.createDataFrame(data_eu, cols_eu)

print("US Data:")
df_us.show()
print("EU Data:")
df_eu.show()

In [0]:
## Answer:
df_us.unionByName(df_eu).display()

### Question 12

**Level:**
Basic

**Question:**
The HR team requested a report of all employees. They want the final output to be sorted logically:

1. First, order the employees alphabetically by their `department` (A to Z).
2. Second, for employees working within the *same* department, order them by `salary` from **highest to lowest**.

Write a PySpark query to achieve this multi-column sorting.

**Practical Use Case:**
Sorting data is a frequent requirement before delivering datasets to business users or writing to sequential files (like CSVs for legacy systems). While sorting inside a distributed system is computationally expensive, it is often a strict business requirement for reporting layers.

**PySpark Query for Table Creation:**

```python
from pyspark.sql.functions import col, desc, asc

data = [
    (1, "Alice", "Sales", 60000),
    (2, "Bob", "Engineering", 120000),
    (3, "Charlie", "Sales", 75000),
    (4, "Diana", "Engineering", 90000),
    (5, "Eve", "HR", 50000),
    (6, "Frank", "Engineering", 105000)
]

columns = ["emp_id", "name", "department", "salary"]

df = spark.createDataFrame(data, columns)
df.display()

```

**Expected Output:**

| emp_id | name | department | salary |
| --- | --- | --- | --- |
| 2 | Bob | Engineering | 120000 |
| 6 | Frank | Engineering | 105000 |
| 4 | Diana | Engineering | 90000 |
| 5 | Eve | HR | 50000 |
| 3 | Charlie | Sales | 75000 |
| 1 | Alice | Sales | 60000 |

*Waiting for your PySpark solution.*

In [0]:
## Question:
from pyspark.sql.functions import col, desc, asc

data = [
    (1, "Alice", "Sales", 60000),
    (2, "Bob", "Engineering", 120000),
    (3, "Charlie", "Sales", 75000),
    (4, "Diana", "Engineering", 90000),
    (5, "Eve", "HR", 50000),
    (6, "Frank", "Engineering", 105000)
]

columns = ["emp_id", "name", "department", "salary"]

df = spark.createDataFrame(data, columns)
df.display()